# LLM evaluation — interview prep

This notebook pairs a **study plan** with **hands-on code** you can extend. Interview loops often expect you to narrate an end-to-end pipeline *and* implement a metric or judge wrapper.

---

## Study plan (2–3 weeks, adjustable)

| Week | Focus | Deliverable |
|------|--------|-------------|
| **1** | **Frame the eval** — task type (QA, summarization, tool-use, RAG), success criteria, failure modes, baseline (human / simpler model). | 1-page spec: inputs, outputs, rubric bullets, and 20–50 golden examples. |
| **1** | **Deterministic metrics** — exact match, token F1, regex constraints, JSON schema validity. | Implement from scratch + explain when they break for generative tasks. |
| **2** | **N-gram / overlap** — BLEU, ROUGE; know limitations for LLM text. | Run `evaluate` or `rouge_score`; compare to reference answers. |
| **2** | **Semantic similarity** — embeddings + cosine similarity; threshold tradeoffs. | Pair predictions with references; discuss false positives on paraphrases. |
| **2** | **LLM-as-judge** — rubric prompts, position bias, scaling, parsing structured scores. | Implement judge prompt + JSON parsing; **always** cross-check a subset with humans. |
| **3** | **Aggregation & rigor** — stratified sampling, confidence intervals, versioning datasets. | Bootstrap CIs; track prompt/dataset git hashes or LangSmith-style runs. |
| **3** | **Ecosystem** — **RAG**: Ragas; **general**: DeepEval, OpenAI Evals patterns, tracing (LangSmith). | One notebook cell each: install + minimal API usage (when you have keys). |

**End-to-end story (say this in interviews):** define task → build/curate golden set → automated checks → semantic / judge layers → human audit on disagreements → aggregate with uncertainty → regression on every change.

---

## Optional installs (run what you need)

```bash
pip install numpy datasets evaluate rouge-score sentence-transformers openai ragas deepeval
```

Core sections below run with **numpy only**; later cells note optional imports.

## 1. Define the task & toy dataset

Golden examples: `(question, reference_answer, model_prediction)`. Swap `prediction` for your model later.

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Any


@dataclass
class EvalExample:
    id: str
    question: str
    reference: str
    prediction: str
    metadata: dict[str, Any] | None = None


EXAMPLES: list[EvalExample] = [
    EvalExample(
        id="capitals-1",
        question="What is the capital of France?",
        reference="Paris",
        prediction="Paris",
    ),
    EvalExample(
        id="capitals-2",
        question="Capital of Japan?",
        reference="Tokyo",
        prediction="The capital is Tokyo.",
    ),
    EvalExample(
        id="math-1",
        question="What is 12 + 28?",
        reference="40",
        prediction="41",
    ),
]

len(EXAMPLES), EXAMPLES[0]

## 2. Normalization + exact match + token F1

Interview classic: implement **EM** and **token-level F1** (SQuAD-style). Discuss casing, punctuation, and articles.

In [ ]:
import re
import string


def normalize_answer(s: str) -> str:
    s = s.lower()
    s = s.translate(str.maketrans("", "", string.punctuation))
    s = re.sub(r"\s+", " ", s).strip()
    return s


def exact_match(pred: str, ref: str) -> bool:
    return normalize_answer(pred) == normalize_answer(ref)


def token_f1(pred: str, ref: str) -> float:
    pred_toks = normalize_answer(pred).split()
    ref_toks = normalize_answer(ref).split()
    if not pred_toks or not ref_toks:
        return float(pred_toks == ref_toks)
    common = 0
    ref_counts: dict[str, int] = {}
    for t in ref_toks:
        ref_counts[t] = ref_counts.get(t, 0) + 1
    pred_counts: dict[str, int] = {}
    for t in pred_toks:
        pred_counts[t] = pred_counts.get(t, 0) + 1
    for t, c in pred_counts.items():
        if t in ref_counts:
            common += min(c, ref_counts[t])
    precision = common / len(pred_toks)
    recall = common / len(ref_toks)
    if precision + recall == 0:
        return 0.0
    return 2 * precision * recall / (precision + recall)


for ex in EXAMPLES:
    print(
        ex.id,
        "EM",
        exact_match(ex.prediction, ex.reference),
        "F1",
        round(token_f1(ex.prediction, ex.reference), 3),
    )

## 3. ROUGE via Hugging Face `evaluate` (optional)

Install: `pip install evaluate rouge-score`. Good for summarization overlap; weak for factual QA alone.

In [ ]:
try:
    import evaluate

    rouge = evaluate.load("rouge")
except Exception as e:
    rouge = None
    print("Skipping ROUGE:", e)

if rouge is not None:
    ex = EXAMPLES[1]
    scores = rouge.compute(
        predictions=[ex.prediction],
        references=[ex.reference],
        use_stemmer=True,
    )
    scores

## 4. Embedding cosine similarity (optional `sentence-transformers`)

Shows **threshold** sensitivity. Without the library, we mock fixed embeddings for the same math.

In [ ]:
import numpy as np


def cosine_sim(a: np.ndarray, b: np.ndarray) -> float:
    denom = (np.linalg.norm(a) * np.linalg.norm(b)) or 1e-12
    return float(np.dot(a, b) / denom)


def embed_texts_mock(texts: list[str], dim: int = 8) -> np.ndarray:
    """Deterministic toy embeddings from character hashes (no ML dependency)."""
    rng = np.random.default_rng(0)
    out = np.zeros((len(texts), dim), dtype=np.float64)
    for i, t in enumerate(texts):
        seed = sum(ord(c) * (j + 1) for j, c in enumerate(t[:200])) % (2**32)
        out[i] = rng.standard_normal(dim, dtype=np.float64) * 0.01 + seed % 7
    return out


pairs = [(ex.prediction, ex.reference) for ex in EXAMPLES]
pred_emb = embed_texts_mock([p for p, _ in pairs])
ref_emb = embed_texts_mock([r for _, r in pairs])
for ex, pe, re in zip(EXAMPLES, pred_emb, ref_emb, strict=True):
    print(ex.id, "cosine(pred, ref)", round(cosine_sim(pe, re), 4))

# Uncomment when sentence-transformers is installed:
# from sentence_transformers import SentenceTransformer
# model = SentenceTransformer("all-MiniLM-L6-v2")
# pred_emb = model.encode([ex.prediction for ex in EXAMPLES])
# ref_emb = model.encode([ex.reference for ex in EXAMPLES])

## 5. LLM-as-judge — prompt template + structured parsing

**Interview talking points:** rubric clarity, anchor examples, position bias (swap A/B), calibration, cost/latency, and validating judges against humans.

Below: a **mock** judge returns fixed scores so the notebook runs offline. Swap `call_judge_model` for OpenAI/Anthropic.

In [ ]:
import json


JUDGE_SYSTEM = """You grade short answers. Output ONLY valid JSON with keys:
correctness (0-1 float), helpfulness (0-1 float), reasoning (one sentence).
Use the reference answer as ground truth but allow paraphrases if factually equivalent."""


def build_judge_user_prompt(question: str, reference: str, prediction: str) -> str:
    return (
        f"Question: {question}\n"
        f"Reference answer: {reference}\n"
        f"Model answer: {prediction}\n"
        "Respond with JSON only."
    )


def parse_judge_json(raw: str) -> dict:
    raw = raw.strip()
    if raw.startswith("```"):
        raw = raw.strip("`").removeprefix("json").strip()
    return json.loads(raw)


def mock_judge_llm(user_prompt: str) -> str:
    """Replace with real API: messages=[system, user], temperature=0."""
    if "41" in user_prompt and "40" in user_prompt:
        return json.dumps(
            {
                "correctness": 0.0,
                "helpfulness": 0.3,
                "reasoning": "Numeric answer wrong vs reference.",
            }
        )
    return json.dumps(
        {
            "correctness": 1.0,
            "helpfulness": 0.9,
            "reasoning": "Matches reference semantically.",
        }
    )


def score_with_judge(ex: EvalExample) -> dict:
    user = build_judge_user_prompt(ex.question, ex.reference, ex.prediction)
    raw = mock_judge_llm(user)
    return parse_judge_json(raw)


[(ex.id, score_with_judge(ex)) for ex in EXAMPLES]

## 6. Pairwise comparison sketch (A vs B, mitigate position bias)

Present **both** orders or randomize; aggregate wins / use Bradley–Terry or Elo for interview depth.

In [ ]:
import random


def pairwise_prompt(question: str, ref: str, a: str, b: str, order: tuple[str, str]) -> str:
    first, second = order
    first_ans, second_ans = (a, b) if order == ("A", "B") else (b, a)
    return (
        f"Question: {question}\nReference (ground truth): {ref}\n"
        f"Answer {first}: {first_ans}\nAnswer {second}: {second_ans}\n"
        "Which answer is better overall (correctness + clarity)? Reply ONLY: A, B, or tie."
    )


def mock_pairwise_judge(prompt: str) -> str:
    """Demo judge: both answers mention Paris → tie (swap-safe). Replace with LLM API."""
    low = prompt.lower()
    if low.count("paris") >= 2:
        return "tie"
    return "A"


def evaluate_pairwise(question: str, ref: str, model_a: str, model_b: str, *, seed: int = 0) -> dict:
    rng = random.Random(seed)
    orders = [("A", "B"), ("B", "A")]
    rng.shuffle(orders)
    votes = []
    for order in orders:
        p = pairwise_prompt(question, ref, model_a, model_b, order)
        v = mock_pairwise_judge(p)
        votes.append((order, v))
    return {"votes": votes}


evaluate_pairwise(
    "Capital of France?",
    "Paris",
    model_a="Paris",
    model_b="paris.",
)

## 7. Aggregate metrics + bootstrap confidence interval

Shows you can discuss **uncertainty** on small eval sets — common senior-level follow-up.

In [ ]:
def dataset_em_f1(examples: list[EvalExample]) -> dict[str, float]:
    ems = [exact_match(ex.prediction, ex.reference) for ex in examples]
    f1s = [token_f1(ex.prediction, ex.reference) for ex in examples]
    return {"exact_match": sum(ems) / len(ems), "token_f1_mean": sum(f1s) / len(f1s)}


def bootstrap_ci_mean(values: list[float], *, n_boot: int = 1000, seed: int = 0, alpha: float = 0.05):
    rng = np.random.default_rng(seed)
    arr = np.asarray(values, dtype=np.float64)
    n = len(arr)
    stats = []
    for _ in range(n_boot):
        sample = rng.choice(arr, size=n, replace=True)
        stats.append(float(sample.mean()))
    stats.sort()
    low = stats[int((alpha / 2) * n_boot)]
    high = stats[int((1 - alpha / 2) * n_boot) - 1]
    return {"mean": float(arr.mean()), "ci_low": low, "ci_high": high}


print("Point estimates:", dataset_em_f1(EXAMPLES))
f1_values = [token_f1(ex.prediction, ex.reference) for ex in EXAMPLES]
print("Bootstrap token-F1 mean CI:", bootstrap_ci_mean(f1_values))

## 8. Library cheat sheet — what to say in interviews

| Library | Typical use |
|---------|----------------|
| **`datasets`** | Load public benchmarks (HF); version splits. |
| **`evaluate`** | BLEU, ROUGE, BLEURT wrappers; fast iteration. |
| **`ragas`** | Faithfulness, answer relevance, context precision/recall for RAG. |
| **`deepeval`** | Metric presets + LLM test cases in CI-ish workflows. |
| **LangSmith / Phoenix** | Trace runs, compare experiments, human annotation queues. |
| **OpenAI Evals** | YAML specs + grading patterns; good mental model for prod eval pipelines. |

**RAG mini-pattern:** retrieve → generate → score *faithfulness* (claim-level vs context) + *answer relevance* (vs question). Mention **chunking**, **top-k**, and **hallucination** checks.

---

### Next steps for you

1. Replace `mock_judge_llm` with a real chat completion (`openai`, etc.), `temperature=0`, log raw outputs.
2. Export `EXAMPLES` to JSONL and version in git; add adversarial cases (negation, dates, units).
3. Pick one public slice (e.g. subset of **TruthfulQA** or **GSM8K**) and run EM/F1 + one semantic metric.
4. Practice whiteboarding: draw data flow from **prompt → model → parser → metric → dashboard**.

In [ ]:
# Optional: skeleton for OpenAI judge (set OPENAI_API_KEY in your environment)
# from openai import OpenAI
# client = OpenAI()
#
# def call_judge_model(user_prompt: str) -> str:
#     resp = client.chat.completions.create(
#         model="gpt-4o-mini",
#         temperature=0,
#         messages=[
#             {"role": "system", "content": JUDGE_SYSTEM},
#             {"role": "user", "content": user_prompt},
#         ],
#     )
#     return resp.choices[0].message.content or ""

"""Ragas requires API + contexts for full RAG metrics; see https://docs.ragas.io """

print("Notebook core sections complete. Enable optional imports above for live API/RAG runs.")